
Tiếp theo hãy giúp tôi xây dựng thêm InventorySystem sao cho:
- Có thể nhặt được nhiều loại đồ khác nhau (Items, Consumables, Materials) và phân biệt chúng trong kho đồ.
- Có thể cập nhật và thay đổi cách chỉ số của vật phẩm khi bị
- Có thể xoá một món đồ cụ thể trong kho đồ dựa trên ID hoặc vị trí (slot) của nó (đặt giả nâng cấp bị thất bại và vũ khí bị phá huỷ)
- Tích hợp các hàm để sau này có thể dễ dàng liên kết với hệ thống Crafting, GemCrafting, Enchanting, .... để nâng cấp đồ hoặc thay đổi chỉ số đồ.
- Tích hợp các hàm để sau này có thể dễ dàng liên kết với hệ thống giao dịch giữa người chơi (Player Trading) hoặc chợ đen (Black Market) để mua bán đồ với nhau.
- Thêm các lệnh để test các chức năng trên trong chat (VD: /addconsumable, /addmaterial, /removeitem, /upgradeitem, /tradeitem, ...)

ItemData:
-- File: Shared/Components/ItemData.luau
local ItemDatabase = require(game.ReplicatedStorage.Shared.Constants.ItemData)
	local HttpService = game:GetService("HttpService")
-- Định nghĩa kiểu dữ liệu 
export type Enchant = {
	Id: string, 
	Level: number
}

-- Mẫu dữ liệu cho 1 lần nâng cấp
export type UpgradeEntry = {
	Material: string,            -- Loại tinh thể/vật liệu dùng
	StatChanges: {[string]: number}, -- Các chỉ số được cộng (VD: {Damage = 2, Crit = 5})
	Time: number            -- Lưu thời điểm để sau này làm log
}


export type ItemData = {
	UUID: string,             -- ID độc nhất của cục đồ này (VD: "item_982hjsdf-2342")
	ID: string,       -- Tên của đồ gốc (VD: "IronSword" -> để tra trong Constants/Weapons)

	-- Các chỉ số bị biến đổi (Chỉ tồn tại khi có sự thay đổi)
	CustomName: string?,      -- Dấu "?" nghĩa là có thể bị nil (không lưu vào DataStore nếu trống)
	CustomLore: string?,

	UpgradeCount: number?,    -- Thay cho UpgradeAttemptsUsed, nếu = 0 thì gán bằng nil luôn
	Purity: number?,
	Corruption: number?,

	OwnerId: number?,         -- UserId của chủ sở hữu (nếu là đồ bị khóa)
	BindState: "Unlocked" | "BoundToCharacter" | "BoundToAccount", -- Dễ đọc, dễ hiểu

	-- Dùng Array để lưu, không giới hạn số slot cứng.
	-- Server sẽ check `#Enchants < MaxEnchantSlots` lấy từ Template gốc.
	Enchants: {Enchant}?,     
	Gems: {string}?,          -- Chỉ cần lưu mảng ID của các viên ngọc (VD: {"Ruby", "Sapphire"})
	UpgradeHistory: {UpgradeEntry}?,
	
	State: {string}?,
	NBT: {[string]: any}?,     -- Lưu data tùy ý (Custom Data) cho các event đặc biệt
	Aura: number?
}

-- Trả về 1 hàm Factory để tạo vật phẩm mới cực kỳ sạch sẽ
local ItemFactory = {}

-- [MỚI] Thêm dấu "?" vào `ItemData?` vì hàm này có thể trả về nil nếu ID nhập vào bị sai
function ItemFactory.createNewItem(Id: string): ItemData?
    
    -- 1. KIỂM TRA TÍNH HỢP LỆ CỦA ID TỪ DATABASE
    if not ItemDatabase.Registry[Id] then
        warn("🚨 [ItemFactory] Từ chối tạo đồ! ID vật phẩm không tồn tại trong GlobalRegistry: " .. tostring(Id))
        return nil
    end

    -- 2. NẾU HỢP LỆ, TIẾN HÀNH TẠO NHỮNG DATA BẮT BUỘC
    local newItem: ItemData = {
        UUID = HttpService:GenerateGUID(false),
        ID = Id,
        BindState = "Unlocked",
    }

    return newItem
end

return ItemFactory

StackableData:
-- File: Shared/Components/StackableData.luau

export type StackableData = {
	Id: string,     -- ID gốc (VD: "HealthPotion", "IronOre")
	Amount: number,         -- Số lượng
	NBT: {[string]: any}?   -- Dành cho trường hợp đặc biệt (ví dụ: bình máu có hạn sử dụng)
}

local StackableFactory = {}

function StackableFactory.create(Id: string, amount: number): StackableData
	return {
		Id = Id,
		Amount = amount or 1,
	}
end

return StackableFactory

UpgradeTemplate:
-- File: Shared/Components/UpgradeTemplate.luau

-- Điều kiện ảnh hưởng tỉ lệ thành công
export type UpgradeCondition = {
	StatName: string,       -- Ví dụ: "Purity", "Corruption"
	RatePerPoint: number,   -- Lệch % mỗi điểm (VD: -0.1 = -10%)
	MaxCap: number,         -- Giới hạn tối đa (VD: -0.9)
}

-- [POOL PHỤ] Roll thông số
export type StatRollEntry = {
	StatName: string,
	Min: number,
	Max: number,
	LevelScale: {Min: number, Max: number}?, -- Dành cho việc scale theo level (thay cho "7:75")
	Weight: number, -- Trọng số trong pool phụ
}

-- [POOL CHÍNH] Các kịch bản có thể xảy ra khi nâng cấp
export type PoolOutcome = {
	Weight: number,                  -- Trọng số của kịch bản này trong pool chính

	-- Danh sách các pool phụ (Sẽ quay random bên trong này tiếp)
	StatPools: {StatRollEntry}?,     

	-- Các hiệu ứng và trạng thái áp dụng thẳng
	ApplyStates: {number}?,          -- Thay cho Value={1,2,3...}
	SpecialEffects: {string}?,       -- VD: {"ResetUpgrades", "ResetPurity"}

	Message: string,
	MessageColor: Color3,
}

-- Khối logic nâng cấp chính
export type UpgradeLogic = {
	AcceptTypes: {string},           -- VD: {"Weapon", "Armor", "All"}
	UpgradeCountCost: number,        -- Tốn bao nhiêu lượt nâng (thường là 1)

	BaseSuccessRate: number,         -- Tỉ lệ thành công gốc (0.0 -> 1.0)
	Conditions: {UpgradeCondition}?, -- Các điều kiện thay đổi tỉ lệ

	-- Các Pool Trọng Số
	SuccessPool: {PoolOutcome},      -- Nếu ép thành công thì roll bảng này
	FailPool: {PoolOutcome}?,        -- Nếu ép xịt thì roll bảng này
}

return {}

```lua
-- ============================================================================
--                      TỪ ĐIỂN QUY CHUẨN HỆ THỐNG ENCHANT & RELIC
--      (Sự kết hợp giữa Expedition 33 - Turn-Based và Slay the Spire 2 - Roguelike)
-- ============================================================================

-- ==========================================
-- 1. TỪ ĐIỂN TRIGGER (KHI NÀO KÍCH HOẠT?)
-- ==========================================

    -- [Trong Trận Đấu (In-Combat Triggers)]
    -- OnCombatStart        : Ngay khi trận đấu vừa nổ ra (Trước khi bất kỳ ai có lượt).
    -- OnTurnStart          : Khi bắt đầu lượt của bản thân nhân vật sở hữu.
    -- OnTurnEnd            : Khi kết thúc lượt của bản thân (Dùng để tính sát thương Độc/Cháy).
    -- OnHit                : Khi tấn công trúng mục tiêu (Bao gồm cả đánh thường và kỹ năng).
    -- OnCrit               : Khi đòn tấn công nổ sát thương Chí Mạng.
    -- OnDamageTaken        : Khi bản thân bị nhận sát thương từ đối thủ.
    -- OnPerfectParry       : Khi bấm nút đỡ đòn (Block/Parry) hoàn hảo đúng khung hình phản xạ.
    -- OnDodge              : Khi né đòn tấn công của đối thủ thành công.
    -- OnActionPointSpent   : Khi tiêu hao năng lượng hoặc điểm AP để tung chiêu thức.
    -- OnStatusChange       : Khi nhận hoặc mất một bùa lợi/hại (Buff/Debuff).
    -- OnKill               : Khi hạ gục một mục tiêu.
    -- OnTeammateDeath      : Khi một đồng minh trong đội hình ngã xuống.
    -- OnDeath              : Khi bản thân hết máu (Dùng cho các bùa Hồi sinh/Tự bạo).

    -- [Khám Phá Bản Đồ (Out-of-Combat / Exploration Triggers)]
    -- OnEliteEncounter     : Ngay khi trận đấu với Quái Tinh Anh (Elite) bắt đầu.
    -- OnBossEncounter      : Ngay khi trận đấu với Boss Khu vực/Cấp độ bắt đầu.
    -- OnCombatEnd          : Khi trận đấu kết thúc với kết quả Chiến thắng.
    -- OnMapMove            : Mỗi khi người chơi di chuyển sang một ô Node mới trên bản đồ hầm ngục.
    -- OnEnterCampfire      : Khi bước vào ô Lửa Trại (Trạm nghỉ ngơi/Rèn đập đồ).
    -- OnEnterShop          : Khi bước vào ô gặp Thương Nhân (Merchant).
    -- OnEnterEvent         : Khi bước vào ô Sự Kiện ẩn (Dấu ? trên bản đồ).
    -- OnTakeMapDamage      : Khi bị mất máu do cạm bẫy hoặc sự kiện lựa chọn ngoài bản đồ.
    -- OnCard/ItemDraft     : Khi hiển thị màn hình chọn phần thưởng (Hệ thống chọn 1 trong 3).


-- ==========================================
-- 2. TỪ ĐIỂN ĐIỀU KIỆN (CONDITIONS)
-- ==========================================

    -- [Kiểm Tra Chỉ Số & Trạng Thái Sinh Tồn]
    -- Chance               : Tỉ lệ % ngẫu nhiên xảy ra hiệu ứng. VD: 0.5 (50%).
    -- TargetHealthBelow    : Máu của mục tiêu hiện tại đang dưới X%.
    -- TargetHealthAbove    : Máu của mục tiêu hiện tại đang trên X% (Dùng cho bùa Đánh phủ đầu).
    -- SelfHealthBelow      : Máu của bản thân hiện tại đang dưới X%.
    -- TargetHasStatus      : Mục tiêu đang bị dính hiệu ứng cụ thể nào đó (Poison, Burn, Freeze...).
    -- SelfHasStatus        : Bản thân đang sở hữu hiệu ứng cụ thể nào đó.
    -- StatusStackAbove     : Mục tiêu đang bị dính cộng dồn hiệu ứng lớn hơn X lần (VD: > 4 stack Độc).

    -- [Kiểm Tra Cơ Chế Lượt (Turn-Based)]
    -- ActionPointsBelow    : Điểm AP/Năng lượng của bản thân đang dưới ngưỡng X.
    -- IsFirstTurn          : Chỉ kích hoạt duy nhất ở lượt đầu tiên khi trận đấu bắt đầu.
    -- IsStaggered          : Mục tiêu đang trong trạng thái bị "Phá vỡ điểm yếu" (Stagger/Break).
    -- HasElementAdvantage  : Đòn tấn công mang thuộc tính khắc chế được Hệ nguyên tố của quái.
    -- ComboCountAbove      : Số lượt hoặc số đòn đánh liên tiếp trúng đích lớn hơn X.

    -- [Kiểm Tra Tiến Trình & Tài Nguyên Run]
    -- GoldAbove            : Số tiền vàng/Mảnh linh hồn tích lũy trong hầm ngục đang có > X.
    -- CurrentMaxHealth     : Máu tối đa (Max HP) hiện tại đang ở ngưỡng X cụ thể.
    -- HasCurseEnchant      : Người chơi đang mang ít nhất 1 bùa nguyền rủa (Curse) trên người.
    -- ConsumableEmpty      : Túi đồ tiêu hao (Bình máu/Mana) đã hết sạch không còn cái nào.
    -- NodeCountAbove       : Tổng số ô Node đã đi qua trên bản đồ lớn hơn X.
    -- IsEliteOrBossRoom    : Ô Node hiện tại đang đứng hoặc sắp đánh là phòng Tinh Anh/Boss.
    -- RestOptionSelected   : Tại ô Lửa trại, người chơi đưa ra lựa chọn là "Ngủ" (Hồi máu).


-- ==========================================
-- 3. TỪ ĐIỂN HÀNH ĐỘNG (ACTIONS)
-- ==========================================

    -- [Tác Động Giao Tranh (In-Combat Actions)]
    -- DealDamage           : Gây sát thương (Có thể cấu hình Physical, Magic, TrueDamage).
    -- Heal                 : Hồi phục máu tức thời trong trận.
    -- ApplyStatus          : Gắn hiệu ứng trạng thái lên mục tiêu (Fear, Burn, Stun, Taunt...).
    -- ModifyStat           : Tăng hoặc giảm một chỉ số chiến đấu tạm thời (Giáp, Tốc độ, Kháng phép...).
    -- AoECleave            : Gây sát thương lan tỏa sang các quái vật đứng cạnh mục tiêu gốc.
    -- Shield               : Lập một lớp khiên ảo (Barrier) chặn X lượng sát thương nhận vào.
    -- CleanseStatus        : Xóa bỏ 1 hoặc toàn bộ hiệu ứng bất lợi đang dính trên cơ thể.
    -- RestoreAP            : Hồi phục điểm hành động (AP/Mana) để có thêm năng lượng ra chiêu.
    -- AdvanceTimeline      : Đẩy nhân vật lên trước trên thanh thứ tự (Initiative) để cướp lượt sớm.
    -- DelayTargetTurn      : Đánh lùi vị trí của quái trên thanh thứ tự, bắt nó phải chờ lượt lâu hơn.
    -- CounterAttack        : Tự động tung đòn phản công ngay lập tức ngoài hiệp đấu.
    -- Revive               : Hồi sinh bản thân hoặc đồng đội bị gục ngã với X% máu.

    -- [Tác Động Tiến Trình Hầm Ngục (Out-of-Combat Actions)]
    -- ModifyMaxHealth      : Tăng hoặc giảm vĩnh viễn giới hạn Máu Tối Đa (Max HP) trong suốt chuyến đi.
    -- MapHeal              : Hồi phục thanh máu ngoài giao tranh (Tính theo % Máu tối đa).
    -- GainGold             : Cộng thêm tiền vàng hầm ngục vào túi tài nguyên của người chơi.
    -- DiscountShop         : Giảm giá toàn bộ các mặt hàng bày bán tại ô Thương Nhân đi X%.
    -- PreBattleShield      : Cấp một lượng Giáp ảo lớn ngay khi bước vào trận đấu tiếp theo.
    -- PreBattleAP          : Cấp thêm điểm hành động bonus ở Hiệp 1 của trận đấu tiếp theo.
    -- RevealMapNode        : Nhìn thấu sương mù bản đồ, hiển thị chính xác các ô ẩn (Dấu ?) tiếp theo là gì.
    -- UpgradeRandom        : Chọn ngẫu nhiên 1 dòng Enchant đang có trên người và nâng cấp lên +1 Cấp.
    -- AddCurse             : Ép người chơi phải nhận 1 Enchant Nguyền Rủa (Đánh đổi lấy lợi ích khác).
```